# Same Claim, Different Script — reproducible evaluation

This notebook validates the final benchmark and runs genuine model inference only when the user supplies a model/checkpoint and credentials. It never fabricates predictions. Few-shot demonstrations must be stored outside the 70-claim benchmark.

In [ ]:
!pip -q install pandas scikit-learn scipy statsmodels transformers accelerate sentencepiece
from pathlib import Path
import json, os, re, time
import pandas as pd
ROOT = Path('/content/health_claim_verification_project')
DATA = ROOT / 'data' / 'final_benchmark_70.csv'
RAW_OUT = ROOT / 'results' / 'raw_outputs.jsonl'
RAW_OUT.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(DATA)
assert df.claim_id.nunique() == 70
assert len(df) == 70
assert df.gold_label.value_counts().to_dict() == {'Supported': 24, 'Refuted': 23, 'Insufficient': 23}
assert df.topic.nunique() == 7
df.head(2)

## Split and few-shot policy

The benchmark is the held-out evaluation set. Do not select demonstrations from it. Provide a separate `fewshot_demonstrations.csv` with one or more reviewed examples per label and the same three forms, or leave the few-shot condition disabled.

In [ ]:
DEMO = ROOT / 'data' / 'fewshot_demonstrations.csv'
if DEMO.exists():
    demos = pd.read_csv(DEMO)
    assert not set(demos.claim_id) & set(df.claim_id), 'Leakage: demonstration IDs overlap with test IDs'
else:
    demos = None
    print('Few-shot disabled: no separate demonstration file supplied.')

## Model configuration

Set `MODEL_ID` to an exact checkpoint. For gated Hugging Face models, authenticate through a Colab secret named `HF_TOKEN`. The default is `None` so opening the notebook does not trigger an unintended model download.

In [ ]:
MODEL_ID = None  # e.g. 'Qwen/Qwen2.5-7B-Instruct'
CONDITIONS = ['zero_shot', 'few_shot', 'translate_then_verify']
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 64
SEED = 42
if MODEL_ID is None:
    print('Inference is pending. Set MODEL_ID and ensure hardware/model access before running.')

In [ ]:
from transformers import pipeline
import torch
if MODEL_ID:
    generator = pipeline('text-generation', model=MODEL_ID, device_map='auto', torch_dtype='auto')
else:
    generator = None

In [ ]:
LABELS = ['Supported', 'Refuted', 'Insufficient']
def normalize_label(text):
    t = ' '.join(str(text).lower().split())
    if t in {'supported','support','true','yes'}: return 'Supported'
    if t in {'refuted','refute','false','no'}: return 'Refuted'
    if t in {'insufficient','insufficient evidence','unknown','uncertain'}: return 'Insufficient'
    return None
def prompt_for(claim, condition, form, demonstrations=None):
    prefix = 'Classify the claim as exactly one of Supported, Refuted, or Insufficient. Return only the label.\n'
    if condition == 'few_shot':
        if demonstrations is None: raise ValueError('Few-shot requested without separate demonstrations')
        shots = '\n'.join(f'Claim: {r[form]}\nLabel: {r.gold_label}' for r in demonstrations.itertuples())
        prefix += shots + '\n'
    if condition == 'translate_then_verify' and form != 'english':
        prefix += 'First translate the claim internally into English, then classify it.\n'
    return prefix + f'Claim: {claim}\nLabel:'

In [ ]:
def run_condition(condition, forms=('english','urdu','roman_urdu')):
    if generator is None: raise RuntimeError('Set MODEL_ID before inference')
    if condition == 'few_shot' and demos is None: raise RuntimeError('Separate demonstrations are required')
    records = []
    for item in df.itertuples():
        for form in forms:
            prompt = prompt_for(getattr(item, form), condition, form, demos)
            started = time.time()
            try:
                result = generator(prompt, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, return_full_text=False)[0]['generated_text']
                error = ''
            except Exception as exc:
                result = ''
                error = repr(exc)
            records.append({'run_id': time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()), 'model_requested': MODEL_ID, 'prompting_condition': condition, 'temperature': TEMPERATURE, 'max_new_tokens': MAX_NEW_TOKENS, 'seed': SEED, 'claim_id': item.claim_id, 'topic': item.topic, 'form': form, 'claim_text': getattr(item, form), 'gold_label': item.gold_label, 'predicted_label': normalize_label(result), 'confidence_value': None, 'confidence_method': 'not collected', 'raw_response': result, 'prompt_text': prompt, 'latency_seconds': time.time()-started, 'error': error})
    with RAW_OUT.open('a', encoding='utf-8') as handle:
        for record in records: handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    return pd.DataFrame(records)


Run each condition only after checking the prompts and model configuration. The following cell is intentionally guarded.

In [ ]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    for condition in CONDITIONS:
        if condition != 'few_shot' or demos is not None:
            run_condition(condition)
else:
    print('No inference executed. Set RUN_INFERENCE=True only when model access is available.')

In [ ]:
if RAW_OUT.exists():
    import sys
    sys.path.insert(0, str(ROOT / 'code'))
    from analysis import run_analysis
    run_analysis(RAW_OUT, ROOT / 'results' / 'analysis', n_bootstrap=2000)
else:
    print('No raw outputs available; analysis remains pending.')